# 9.10 Knowledge distillation for LMs (DistilBERT)

Distillation trains a smaller model to imitate the probability shape of a larger teacher.

We use CPU-only logistic teachers and students to implement softened probabilities, KL, cross-entropy, and mistake filtering. No large models are downloaded.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

SEED = 910
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)


def softmax(logits):
    logits = np.asarray(logits, dtype=float)
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / np.sum(exp_values, axis=-1, keepdims=True)


def build_distillation_ladder():
    rng = np.random.default_rng(SEED)
    ladders = []
    ladders.append({"name": "D1 one prompt", "teacher_logits": np.array([[2.0, 1.0, 0.0]]), "student_logits": np.array([[1.2, 0.8, 0.2]]), "human": np.array([0])})

    X_d2 = rng.normal(size=(18, 4))
    W_teacher_d2 = np.array([[1.1, -0.6, 0.2], [-0.2, 0.9, -0.4], [0.4, 0.1, 0.7], [0.3, -0.5, 0.8]])
    teacher_d2 = X_d2 @ W_teacher_d2
    student_d2 = teacher_d2 * 0.65 + rng.normal(scale=0.20, size=teacher_d2.shape)
    human_d2 = np.argmax(teacher_d2, axis=1)
    ladders.append({"name": "D2 few-shot set", "teacher_logits": teacher_d2, "student_logits": student_d2, "human": human_d2})

    X_d3 = rng.normal(size=(28, 5))
    W_teacher_d3 = rng.normal(size=(5, 3))
    teacher_d3 = X_d3 @ W_teacher_d3
    human_d3 = np.argmax(teacher_d3, axis=1)
    teacher_d3[::7] = teacher_d3[::7][:, [1, 0, 2]]
    student_d3 = teacher_d3 * 0.55 + rng.normal(scale=0.35, size=teacher_d3.shape)
    ladders.append({"name": "D3 teacher mistakes", "teacher_logits": teacher_d3, "student_logits": student_d3, "human": human_d3})

    texts = ["helpful budget answer", "unsafe medical claim", "short greeting", "policy summary", "creative idea", "legal qualifier", "pricing math", "irrelevant joke"]
    X_d4 = np.array([
        [1, 1, 0, 0],
        [0, 0, 1, 1],
        [1, 0, 0, 0],
        [1, 0, 1, 0],
        [1, 1, 0, 0],
        [0, 0, 1, 1],
        [1, 1, 0, 0],
        [0, 1, 0, 1]
    ], dtype=float)
    W_d4 = np.array([[1.0, -0.3, 0.1], [0.2, 0.6, -0.4], [-0.5, 0.2, 1.0], [-0.2, -0.1, 0.8]])
    teacher_d4 = X_d4 @ W_d4
    student_d4 = teacher_d4 * 0.60 + rng.normal(scale=0.25, size=teacher_d4.shape)
    human_d4 = np.argmax(teacher_d4, axis=1)
    ladders.append({"name": "D4 text corpus", "teacher_logits": teacher_d4, "student_logits": student_d4, "human": human_d4})

    X_d5 = rng.normal(size=(72, 7))
    W_d5 = rng.normal(scale=0.55, size=(7, 3))
    teacher_d5 = X_d5 @ W_d5
    human_d5 = np.argmax(teacher_d5, axis=1)
    mistake_rows = np.arange(0, len(human_d5), 9)
    teacher_d5[mistake_rows] = teacher_d5[mistake_rows][:, [2, 1, 0]]
    student_d5 = teacher_d5 * 0.50 + rng.normal(scale=0.45, size=teacher_d5.shape)
    ladders.append({"name": "D5 compounding mistakes", "teacher_logits": teacher_d5, "student_logits": student_d5, "human": human_d5, "mistakes": mistake_rows})
    return ladders


## The concept, built once (D1)

The distillation objective is $$\mathcal{L}=T^2\,\mathrm{KL}(\sigma(z_T/T)\Vert\sigma(z_S/T)).$$
The lesson numbers are: teacher logits $[2,1,0]$ give $[0.665,0.245,0.090]$ at $T=1$, $[0.506,0.307,0.186]$ at $T=2$, hard-label CE $-\log(0.7)=0.357$, KL from $[0.7,0.2,0.1]$ to $[0.5,0.3,0.2]$ is $0.085$, and $110/340=32.4\%$.

In [ ]:

def distill_step(teacher_logits, student_logits, temperature=2.0):
    teacher_probs = softmax(teacher_logits / temperature)
    student_probs = softmax(student_logits / temperature)
    kl_terms = teacher_probs * (np.log(teacher_probs + 1e-12) - np.log(student_probs + 1e-12))
    kl = float(np.mean(np.sum(kl_terms, axis=1)))
    scaled_loss = (temperature ** 2) * kl
    return teacher_probs, student_probs, kl, scaled_loss

teacher_probs_t1 = softmax(np.array([[2.0, 1.0, 0.0]]))[0]
teacher_probs_t2 = softmax(np.array([[2.0, 1.0, 0.0]]) / 2.0)[0]
kl_demo = float(np.sum(np.array([0.7, 0.2, 0.1]) * (np.log(np.array([0.7, 0.2, 0.1])) - np.log(np.array([0.5, 0.3, 0.2])))))
ce_demo = -math.log(0.7)
param_fraction = 110 / 340
assert np.allclose(np.round(teacher_probs_t1, 3), np.array([0.665, 0.245, 0.090]))
assert np.allclose(np.round(teacher_probs_t2, 3), np.array([0.506, 0.307, 0.186]))
assert round(ce_demo, 3) == 0.357
assert round(kl_demo, 3) == 0.085
assert round(param_fraction * 100, 1) == 32.4
print(teacher_probs_t1, teacher_probs_t2, ce_demo, kl_demo, param_fraction)


The same function trains against full distributions, not just the top class. That lets us measure both KL-to-teacher and human-label accuracy.

In [ ]:

teacher_probs, student_probs, kl_value, scaled_loss = distill_step(np.array([[2.0, 1.0, 0.0]]), np.array([[1.2, 0.8, 0.2]]), temperature=2.0)
print("teacher", teacher_probs[0])
print("student", student_probs[0])
print("KL", kl_value)
print("T^2 KL", scaled_loss)


## The dataset ladder

The F8 ladder grows from one logit vector to compounding teacher mistakes. Labels named `human` represent held-out checks used to filter teacher errors.

In [ ]:

ladder = build_distillation_ladder()
for rung in ladder:
    print(rung["name"], "examples", rung["teacher_logits"].shape[0], "classes", rung["teacher_logits"].shape[1])
    print("teacher sample", rung["teacher_logits"][0])
    print("student sample", rung["student_logits"][0])


## Run the same distillation loss across D1-D5

In [ ]:

results = []
for rung in ladder:
    teacher_probs, student_probs, kl_value, scaled_loss = distill_step(rung["teacher_logits"], rung["student_logits"], temperature=2.0)
    predictions = np.argmax(student_probs, axis=1)
    accuracy = float(np.mean(predictions == rung["human"]))
    results.append({"name": rung["name"], "teacher": teacher_probs, "student": student_probs, "kl": kl_value, "metric": accuracy})

print("rung | student_accuracy | KL_to_teacher")
for result in results:
    print(f"{result['name']} | {result['metric']:.3f} | {result['kl']:.3f}")


## Results visualization

Small multiples compare teacher and student probabilities for one example per rung; the summary curve tracks student accuracy.

In [ ]:

fig, axes = plt.subplots(1, len(results), figsize=(16, 3))
for ax, result in zip(axes, results):
    width = 0.35
    positions = np.arange(result["teacher"].shape[1])
    ax.bar(positions - width / 2, result["teacher"][0], width=width, label="teacher")
    ax.bar(positions + width / 2, result["student"][0], width=width, label="student")
    ax.set_ylim(0.0, 1.0)
    ax.set_title(result["name"].split()[0])
axes[0].legend()
plt.tight_layout()

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(range(1, len(results) + 1), [item["metric"] for item in results], marker="o")
ax.set_xticks(range(1, len(results) + 1))
ax.set_xticklabels(["D1", "D2", "D3", "D4", "D5"])
ax.set_ylabel("student accuracy")
ax.set_title("Distillation must be checked against human labels")
plt.show()


## Pitfall on the hardest rung

Pitfall: a student can imitate teacher mistakes. On D5, compare imitation-only accuracy with a held-out human-label filter that removes known teacher-mistake rows.

In [ ]:

d5 = ladder[-1]
teacher_labels = np.argmax(d5["teacher_logits"], axis=1)
student_labels = np.argmax(d5["student_logits"], axis=1)
imitation_accuracy = float(np.mean(student_labels == teacher_labels))
human_accuracy = float(np.mean(student_labels == d5["human"]))
valid = teacher_labels == d5["human"]
filtered_human_accuracy = float(np.mean(student_labels[valid] == d5["human"][valid]))
print("imitating teacher", imitation_accuracy)
print("against human labels", human_accuracy)
print("after mistake filter", filtered_human_accuracy)
print("filtered examples", int(np.sum(valid)), "of", len(valid))


## Evaluate it + Practice

- Metric: student accuracy, with KL-to-teacher as a secondary diagnostic.
- No-skill baseline: train on hard teacher top labels only.
- Cheap sanity check: temperature $T=2$ must soften $[2,1,0]$ to $[0.506,0.307,0.186]$.
- Ablation: remove soft labels and observe loss of dark-knowledge information.
- Failure signals: low KL paired with poor human accuracy, especially on known teacher mistakes.

Practice prompts:


**Practice.** Change temperature from 1 to 4 and inspect KL scale.

**Practice.** Filter more D5 teacher mistakes and compare accuracy.

**Practice.** Construct a student that has lower KL but worse top-1 accuracy.